In [1]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode

# MessagesState = {"messages": [...]}; new messages are appended, not overwritten.
# This is what lets the LLM see tool results on the next loop iteration.

In [2]:
from langchain_core.tools import tool


@tool
def add(a: float, b: float) -> float:
    """Add two numbers and return the sum."""   # the docstring is what the LLM reads
    return a + b


tools = [add]

In [3]:
import os
from dataclasses import dataclass
from langchain_ollama import ChatOllama


@dataclass
class LLMConfig:
    model: str = os.getenv("OLLAMA_MODEL", "qwen3:0.6b")        # must support tools, run `ollama pull <model>` first
    base_url: str = os.getenv("OLLAMA_HOST", "http://localhost:11434")
    temperature: float = 0.0
    num_ctx: int = 4096        # context window
    num_predict: int = 512     # max tokens to generate
    timeout: float = 120       # seconds (client timeout)


def build_llm(cfg: LLMConfig) -> ChatOllama:
    return ChatOllama(
        model=cfg.model,
        base_url=cfg.base_url,
        temperature=cfg.temperature,
        num_ctx=cfg.num_ctx,
        num_predict=cfg.num_predict,
        client_kwargs={"timeout": cfg.timeout},
    )


llm_config = LLMConfig()
llm = build_llm(llm_config)

# Tell the model which tools exist. It decides per question whether to use them.
llm_with_tools = llm.bind_tools(tools)


def call_llm(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    # If response.tool_calls is non-empty, the LLM chose to use a tool.
    return {"messages": [response]}

def tools_condition(state):
    last = state["messages"][-1]
    if last.tool_calls:      # the LLM asked for a tool
        return "tools"
    return END               # plain answer -> finish


In [4]:
builder = StateGraph(MessagesState)
builder.add_node("call_llm", call_llm)
builder.add_node("tools", ToolNode(tools))      # executes whatever tool the LLM asked for

builder.add_edge(START, "call_llm")

# tools_condition: if the last message has tool_calls -> "tools", else -> END
builder.add_conditional_edges("call_llm", tools_condition)

# After the tool runs, go back to the LLM so it can use the result (the loop).
builder.add_edge("tools", "call_llm")

In [5]:
graph = builder.compile()

# Small models need a nudge to actually use tools.
SYSTEM = (
    "You are a helpful assistant. You have an `add` tool. For ANY request to add numbers "
    "you MUST call the add tool and never compute the sum yourself. "
    "For other questions answer directly."
)

for question in ["What is 48213.5 plus 99871.25?", "What is a sun? Answer in one sentence."]:
    print("=" * 60)
    print("Q:", question)
    result = graph.invoke({"messages": [("system", SYSTEM), ("user", question)]})
    # print(result)
    for m in result["messages"]:
        m.pretty_print()   # shows human -> AI (tool call) -> tool result -> AI answer

Q: What is 48213.5 plus 99871.25?
================================ System Message ================================

You are a helpful assistant. You have an `add` tool. For ANY request to add numbers you MUST call the add tool and never compute the sum yourself. For other questions answer directly.
================================ Human Message =================================

What is 48213.5 plus 99871.25?
================================== Ai Message ==================================
Tool Calls:
  add (2d6dd434-6509-428e-936a-62412e546d31)
 Call ID: 2d6dd434-6509-428e-936a-62412e546d31
  Args:
    a: 48213.5
    b: 99871.25
================================= Tool Message =================================
Name: add

148084.75
================================== Ai Message ==================================

148084.75
Q: What is a sun? Answer in one sentence.
================================ System Message ================================

You are a helpful assistant. You have an `add

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))